# Text Extraction

we need to create a dictionnary 

key : n° volume, n° article

value : content (title and substract)

In [1]:
import nltk
import pandas as pd
from bs4 import BeautifulSoup
import requests
import os
import time


volume number is inside : #main > div > div > ul > li:nth-child(1) > h2 > span:nth-child(1)

 issue number is inside : #main > div > div > ul > li:nth-child(1) > ul > li:nth-child(1) > a

 href of issue containing articles : #main > div > div > ul > li:nth-child(1) > ul > li:nth-child(1) > a['href']

 link to article : #main > div > div > div > section > ol > li:nth-child(1) > article > div.app-card-open__main > h3 > a
 
 function that can extract title and abstract

In [2]:
# link to extract from
link = "https://link.springer.com/journal/12065/volumes-and-issues"

# the dictionnay will be containing :
# key   : volume number, article number
# value : content of the article (title and absract)
articles_dict = {}


In [3]:
BASE_URL = "https://link.springer.com"
MAIN_URL = f"{BASE_URL}/journal/12065/volumes-and-issues"

def extract_all_articles(save_dir="articles_data"):
    os.makedirs(save_dir, exist_ok=True)
    articles_dict = {}

    # Step 1️⃣: Load main page with all volumes
    main_html = requests.get(MAIN_URL).text
    main_soup = BeautifulSoup(main_html, "html.parser")

    # Each volume = one <li> under #main > div > div > ul
    volume_items = main_soup.select("#main > div > div > ul > li")

    print(f"📚 Found {len(volume_items)} volumes.")

    for v_index, volume_item in enumerate(volume_items, start=1):
        # Extract volume number
        volume_span = volume_item.select_one("h2 > span:nth-child(1)")
        if not volume_span:
            continue
        volume_number = volume_span.get_text(strip=True)
        print(f"\n🔵 Processing {volume_number} ({v_index}/{len(volume_items)})")

        articles_dict[volume_number] = {}

        # Find all issues under this volume
        issue_links = volume_item.select("ul > li > a")
        print(f"  Found {len(issue_links)} issues in {volume_number}.")

        for issue_index, issue_a in enumerate(issue_links, start=1):
            issue_number = issue_a.get_text(strip=True)
            issue_href = issue_a.get("href")
            issue_url = f"{BASE_URL}{issue_href}"
            print(f"  🟢 Issue {issue_number} ({issue_index}/{len(issue_links)}) → {issue_url}")

            # Step 2️⃣: Visit issue page
            try:
                issue_html = requests.get(issue_url).text
                issue_soup = BeautifulSoup(issue_html, "html.parser")
            except Exception as e:
                print(f"  ⚠️ Error loading issue page: {e}")
                continue

            # Step 3️⃣: Extract all article links
            article_links = issue_soup.find_all("a", href=lambda x: x and x.startswith("/article/"))
            print(f"    Found {len(article_links)} articles in {issue_number}.")

            issue_articles = []

            for i, link in enumerate(article_links, start=1):
                href = link.get("href")
                title = link.get_text(strip=True)

                # Skip empty or invalid titles
                if not title:
                    continue

                article_url = f"{BASE_URL}{href}"
                print(f"    🔹 Article {i}: {title}")

                try:
                    article_html = requests.get(article_url).text
                    article_soup = BeautifulSoup(article_html, "html.parser")

                    abstract_div = article_soup.find("div", {"class": "c-article-section__content"})
                    abstract = abstract_div.find("p").get_text(strip=True) if abstract_div else "Abstract not found"

                    issue_articles.append({
                        "title": title,
                        "abstract": abstract
                    })

                    # Save to file
                    file_name = f"{volume_number.replace(' ', '_')}_{issue_number.replace(' ', '_')}_article_{i}.txt"
                    file_path = os.path.join(save_dir, file_name)
                    with open(file_path, "w", encoding="utf-8") as f:
                        f.write(f"Title: {title}\n\nAbstract:\n{abstract}\n")

                except Exception as e:
                    print(f"    ⚠️ Error fetching article: {e}")

                # polite delay
                time.sleep(2)

            # Step 4️⃣: Store results in the dictionary
            articles_dict[volume_number][issue_number] = issue_articles

    print("\n🎉 Extraction complete! All data saved in dictionary and text files.")
    return articles_dict


In [4]:
data = extract_all_articles() 

📚 Found 18 volumes.

🔵 Processing Volume 18 (1/18)
  Found 6 issues in Volume 18.
  🟢 Issue Issue 6 (1/6) → https://link.springer.com/journal/12065/volumes-and-issues/18-6
    Found 4 articles in Issue 6.
    🔹 Article 1: Solution of frequency deviation and multi objective probabilistic optimal power flow of transmission network incorporating UPFC controller using driving training based optimization
    🔹 Article 2: Improved two-archive evolutionary algorithm for constrained multi-objective optimization of WWTP based on intergenerational information guidance
    🔹 Article 3: Privacy-preserving data aggregation in WBNAs using neuro-evolutionary algorithms and post-quantum homomorphic encryption
    🔹 Article 4: An evolutionary algorithm tailored to the quadratic assignment problem
  🟢 Issue Issue 5 (2/6) → https://link.springer.com/journal/12065/volumes-and-issues/18-5
    Found 21 articles in Issue 5.
    🔹 Article 1: A comprehensive review of optimization approaches of district coolin

In [7]:
# saving the data
import pickle
with open("data/articles_dict.pkl", "wb") as f:
    pickle.dump(data, f)


# Text Display part 2

Define ways to display
- either by article : give number of article and volume
- either by volume : dipsplay all articles

In [ ]:
#  load data .pkl from data folder
data = 


In [ ]:
# option = ""
# case option:
#   "display_by_volume:
#     volume = input
#     #  access all articles of volume 
#     content = data where volume_number = volume
#     display content (title and abstract)
#   "display_by_article":
#     article = input, volume = input
#     content = data where n°volume and n°article match
#     display content

# Part 3 Text Preprocessing

In [ ]:
# for each article extract tokens and add as a item in the dictionnary



the vision of the dictionnary:

volume:
    issue:
        article:
            title: ".."
            abstract: ".."
            tokens: [...]
            normalised tokens snow: [...]
            normalised tokens porter: [...]
            normalised tokens lancaster: [...]


In [ ]:
# for each article and for all its tokens, apply stemming 
# and add the results into another item in the dictionnay
# the stemmer will be different : snow, porte stemmer and lancaster


# Part 4: N-Gram Language Model

In [ ]:
# three models are going to be deployed : unigram, bigram, trigram
# for each article
#   for each model, access tokens threw the dictionnary*
#     create a folder of the model
#     for each token generate its frequency and probability 
#       add the results to a new dictionnary article_modeltype containing token as a key and (frequency, probability) as the value
#      save the dictiornnay to the folder 